# USGS Water Data client (`georest.restusgs.waterdata`) — usage examples

Walks through every public function in `src/georest/restusgs/waterdata.py` against the live
USGS Water Data OGC API (`https://api.waterdata.usgs.gov/ogcapi/v1`):

- `list_collections`, `get_queryables` — what the API offers
- `search_monitoring_locations`, `get_monitoring_location` — find sites
- `get_time_series_metadata` — what a site measures
- `get_daily_values`, `get_continuous_values`, `get_latest_values` — observations
- `get_field_measurements`, `get_peaks` — site visits and annual peaks
- `get_items` — any collection, any filter
- `to_rows` — flatten to plain dicts (CSV / pandas ready)
- API keys, `max_items` / `truncated`, `rate_limit`, and error handling

Every function returns a GeoJSON `FeatureCollection` dict — one Feature per **observation**
for the time-series collections, one per site for `monitoring-locations`.

**Rate limits.** Anonymous callers get "a few queries per hour". This notebook makes ~20
requests, so set `USGS_API_KEY` first (free key: https://api.waterdata.usgs.gov/signup/) or
run the cells selectively.

## Setup

This notebook needs `georest` installed:

```
pip install georest
```

or, from a clone of this repository, `pip install -e .` from the repo root.

In [ ]:
import json
import os

import georest
from georest.restusgs import waterdata as wd

print("georest", georest.__version__)
print("base URL", wd.BASE_URL)

# Or set it for this process only (never commit a key):
# wd.set_api_key("your-key-here")

def show(fc, n=3):
    """Print the parts of a FeatureCollection that matter when eyeballing a result."""
    feats = fc.get("features", [])
    print(f"{len(feats)} features"
          + ("  [TRUNCATED — server had more]" if fc.get("truncated") else "")
          + (f"  rate_limit={fc['rate_limit']}" if fc.get("rate_limit") else ""))
    for f in feats[:n]:
        print(" ", f.get("id"), json.dumps(f.get("properties"), default=str)[:200])

Lees Ferry on the Colorado River (`USGS-09380000`), gauged continuously since 1921, is the
fixture site throughout. Bare ids (`"09380000"`) get the `USGS-` prefix automatically.

In [ ]:
SITE = "USGS-09380000"
LEES_FERRY_BBOX = (-111.7, 36.8, -111.5, 37.0)   # minx, miny, maxx, maxy (lon/lat)

## 1. `list_collections` and `get_queryables` — what the API offers

`COLLECTIONS` is the module's reference list of the data collections; `list_collections()`
is the live, authoritative catalog (it also includes small code tables). `get_queryables`
tells you every property that works as a direct filter on a collection.

In [ ]:
cols = wd.list_collections()
for c in cols:
    print(f"{c['id']:32s} {c.get('title', '')}")

In [ ]:
print(wd.COLLECTIONS)

In [ ]:
q = wd.get_queryables("daily")
print(len(q), "queryable properties on 'daily'")
for name in list(q)[:15]:
    print(f"  {name:36s} {q[name].get('type', '')}")

## 2. `search_monitoring_locations` — find sites

Filters combine with AND. `site_type_code`: `ST` stream, `LK` lake, `GW` well, `SP` spring…
A list of codes becomes a CQL2 `IN`. `hydrologic_unit_code` prefix-matches.

In [ ]:
# Streams inside a bounding box
fc = wd.search_monitoring_locations(bbox=LEES_FERRY_BBOX, site_type_code="ST")
show(fc)

In [ ]:
# Every stream and lake site in a HUC4 (prefix match), capped at 200
fc = wd.search_monitoring_locations(hydrologic_unit_code="1407",
                                    site_type_code=["ST", "LK"], max_items=200)
show(fc, n=5)

In [ ]:
# By state + county FIPS, trimming the response to a few properties
fc = wd.search_monitoring_locations(state_code="49", county_code="025",
                                    properties=["monitoring_location_name", "site_type"],
                                    max_items=20)
show(fc, n=5)

## 3. `get_monitoring_location` — one site

Returns a single GeoJSON Feature (not a collection): full site attributes and its Point.

In [ ]:
site = wd.get_monitoring_location("09380000")
print(site["id"], site["geometry"])
for k in ("monitoring_location_name", "site_type", "state_name", "county_name",
          "drainage_area", "altitude", "vertical_datum", "hydrologic_unit_code"):
    print(f"  {k:28s} {site['properties'].get(k)}")

## 4. `get_time_series_metadata` — what a site measures

One Feature per (parameter, statistic, computation) with `begin`/`end` dates and the
`time_series_id`. Check this before pulling values so you ask for a parameter the site
actually reports.

In [ ]:
fc = wd.get_time_series_metadata(monitoring_location_id=SITE)
for f in fc["features"]:
    p = f["properties"]
    print(f"{p.get('parameter_code')}  {p.get('parameter_name', '')[:40]:40s} "
          f"{p.get('statistic_id')}  {p.get('computation_period_identifier', ''):12s} "
          f"{p.get('begin')} → {p.get('end')}")

## 5. `get_daily_values` — daily statistics (old `/nwis/dv`)

Defaults: `parameter_code="00060"` (discharge, ft³/s), `statistic_id="00003"` (daily mean).
`start`/`end` accept strings, `date`/`datetime` objects, or an open side. Output is sorted by
`(time_series_id, time)` **client-side** — the API does not sort. Geometry is skipped by
default (the same point on every row).

In [ ]:
fc = wd.get_daily_values(SITE, "00060", start="2024-01-01", end="2024-01-31")
show(fc)
print("first:", fc["features"][0]["properties"]["time"],
      " last:", fc["features"][-1]["properties"]["time"])

In [ ]:
import datetime as dt

# date objects, open-ended range (everything since), daily max + min water temperature
# via a list of statistics. (Section 4 shows which daily series a site has — Lees Ferry
# reports gage height only as 15-minute points, so 00065 here would return 0 features.)
fc = wd.get_daily_values(SITE, "00010", statistic_id=["00001", "00002"],
                         start=dt.date(2024, 6, 1), end=None, max_items=100)
show(fc)

In [ ]:
# Several sites at once → CQL2 IN, one call
fc = wd.get_daily_values(["09380000", "09402500"], start="2024-03-01", end="2024-03-07")
show(fc, n=4)
sites = {f["properties"]["monitoring_location_id"] for f in fc["features"]}
print("sites in result:", sites)

## 6. `to_rows` — flatten for CSV / pandas / plotting

One plain dict per Feature. `value` (and anything else named in `numeric`) becomes a
`float`, or `None` when the API sent something non-numeric (e.g. an `"Ice"` marker).

In [ ]:
fc = wd.get_daily_values(SITE, "00060", start="2024-01-01", end="2024-01-10")
rows = wd.to_rows(fc)
rows[:3]

In [ ]:
# CSV with the stdlib
import csv
import io

buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=["time", "value", "unit_of_measure", "approval_status"],
                        extrasaction="ignore")
writer.writeheader()
writer.writerows(rows)
print(buf.getvalue()[:400])

In [ ]:
# pandas, if you have it (not a georest dependency)
try:
    import pandas as pd
    df = pd.DataFrame(rows).set_index("time")
    display(df[["value", "approval_status"]].head())
    print(df["value"].describe())
except ImportError:
    print("pandas not installed — skip")

## 7. `get_continuous_values` — sensor readings (old `/nwis/iv`)

Typically every 15 minutes, so **~35 000 rows per year per parameter**. The default
`max_items=10_000` covers roughly 100 days; the result says `truncated: True` when it bit.
`start="P2D"` is an ISO 8601 duration — the last two days. `time` carries a UTC offset.

In [ ]:
fc = wd.get_continuous_values(SITE, "00065", start="P2D")
show(fc)
rows = wd.to_rows(fc)
print("first:", rows[0]["time"], rows[0]["value"], " last:", rows[-1]["time"], rows[-1]["value"])

In [ ]:
# Deliberately small cap to see the truncated flag
fc = wd.get_continuous_values(SITE, "00060", start="P7D", max_items=50)
show(fc, n=1)
print("truncated:", fc.get("truncated"))

## 8. `get_latest_values` — most recent observation per series

`source="daily"` (default) or `"continuous"`. No site is required: `bbox` +
`parameter_code` gives current conditions at every gage in an area, so geometry is kept by
default here (useful for a map).

In [ ]:
fc = wd.get_latest_values(SITE)
show(fc, n=10)

In [ ]:
# Current discharge at every gage in a box (continuous → the newest 15-minute reading)
fc = wd.get_latest_values(source="continuous", parameter_code="00060",
                          bbox=(-112.5, 36.5, -111.0, 37.5))
for f in fc["features"]:
    p = f["properties"]
    print(f"{p['monitoring_location_id']:16s} {p['time']}  {p['value']:>10s} {p['unit_of_measure']}  "
          f"{f['geometry']['coordinates']}")

## 9. `get_field_measurements` and `get_peaks`

Field measurements are the manual site-visit readings (groundwater levels, gage height and
discharge measurements used to calibrate sensors). Peaks are the annual maximum streamflow,
one per water year, with a `qualifier` list such as `["REGULATED"]`.

In [ ]:
fc = wd.get_field_measurements(SITE, start="2023-01-01", max_items=20)
show(fc)

In [ ]:
fc = wd.get_peaks(SITE, start="2010-01-01")
for r in wd.to_rows(fc):
    print(r["water_year"], r["time"], f"{r['value']:>10,.0f} ft³/s", r.get("qualifier"))

## 10. `get_items` — any collection, any filter

The generic fetch under every helper. Use it for a collection without a helper, or with a
CQL2 `filter`. Any queryable property works as a keyword equality filter. Unlike the helpers,
this path does *not* add `time`/`value` back when you trim with `properties=`.

In [ ]:
# A code table
fc = wd.get_items("agency-codes", max_items=10)
show(fc, n=10)

In [ ]:
# CQL2 text filter + a direct property filter on the daily collection
fc = wd.get_items("daily", monitoring_location_id=SITE, parameter_code="00060",
                  filter="statistic_id IN ('00001','00003') AND approval_status='Approved'",
                  start="2024-01-01", end="2024-01-05", skip_geometry=True,
                  properties="time,statistic_id,value")
show(fc, n=10)

## 11. API keys, `rate_limit`, and errors

Resolution per call: explicit `api_key=` (`""` forces anonymous) → `set_api_key(...)` →
`USGS_API_KEY` / `USGS_WATERDATA_API_KEY` in the environment → anonymous. The key is sent
**only** as the `X-Api-Key` header — never in a URL, so it cannot appear in an error message.

When the API reports the remaining hourly budget, results carry it in-band as `rate_limit`.

In [ ]:
fc = wd.get_daily_values(SITE, start="2024-01-01", end="2024-01-02")
print("rate_limit:", fc.get("rate_limit"))   # None when the server sent no X-RateLimit headers

In [ ]:
# A bad request surfaces the API's own description in the RuntimeError message
try:
    wd.get_daily_values(SITE, start="not-a-date")
except RuntimeError as exc:
    print(type(exc).__name__)
    print(exc)

In [ ]:
# Reserved / invalid arguments are rejected before any request
for bad in (lambda: wd.get_daily_values(SITE, cursor="x"),
            lambda: wd.get_daily_values(SITE, page_size=0),
            lambda: wd.get_latest_values(source="hourly"),
            lambda: wd.get_daily_values([])):
    try:
        bad()
    except (TypeError, ValueError) as exc:
        print(f"{type(exc).__name__}: {exc}")

In [ ]:
# A 429 the client won't wait out raises RateLimitError (a RuntimeError subclass)
# with .retry_after — catch it by name and back off rather than looping.
try:
    fc = wd.get_daily_values(SITE, start="2020-01-01", end="2020-01-02")
    print("ok:", len(fc["features"]), "rows")
except wd.RateLimitError as exc:
    print("rate limited; retry after", exc.retry_after, "seconds")

## 12. Sanity check: the client-side sort

The API returns rows in storage order (a January window can come back starting on the
22nd). Every time-series helper sorts by `(time_series_id, time)`; `get_items` does not.

In [ ]:
raw = wd.get_items("daily", monitoring_location_id=SITE, parameter_code="00060",
                   statistic_id="00003", start="2020-01-01", end="2020-01-31",
                   skip_geometry=True, properties="time,value")
helper = wd.get_daily_values(SITE, "00060", start="2020-01-01", end="2020-01-31")
print("get_items order  :", [f["properties"]["time"] for f in raw["features"][:5]])
print("helper order     :", [f["properties"]["time"] for f in helper["features"][:5]])